# 10 mises en situation SQL à la banque Beobank

**Vous êtes analyste SQL chez Beobank. Chaque jour, un collègue vient vous voir avec une
question. Vous devez répondre en SQL, sur les vraies données de la banque.**

Ce notebook n'est pas un cours théorique sur la syntaxe SQL (voir `Cours_SQL.ipynb` pour
ça). C'est une série de **10 petites histoires**, dans l'esprit de
`Mises_En_Situation_Bancaires.ipynb` (la version Pandas), mais en SQL.

## Petit glossaire des colonnes du dataset

| Colonne | Table | Ce qu'elle représente |
|---|---|---|
| `NUM_TIE` / `IDT_PI` | TIE, TIE_ADR, TIE_X_CTR | Numéro qui identifie un client de façon unique |
| `COD_TYP_TIE` | TIE | Type de client : 1 = particulier, 2 = entreprise |
| `DAT_NAI` | TIE | Date de naissance du client |
| `COD_LNG_CTR` | TIE | Langue de contact du client (FR ou NL) |
| `IDT_AC` | CTR, TIE_X_CTR, TXN_X_CTR | Numéro qui identifie un compte bancaire de façon unique |
| `DAT_OUV_CTR` / `DAT_CLO_CTR` | CTR | Date d'ouverture / de clôture du compte |
| `COD_ECV_CTR` | CTR | Code d'état du compte : 4 = actif, 6 = clôturé |
| `COD_DEV` | CTR | Devise du compte (EUR, USD...) |
| `SLD_CTR` | CTR | Solde du compte (l'argent qu'il y a dessus) |
| `ADR_EMA` | TIE_ADR | Adresse email du client |
| `NUM_TEL_MOB_INL` / `NUM_TEL_DOM_INL` | TIE_ADR | Téléphone mobile / fixe du client |
| `LIB_OPE_INL_1` | TXN_X_CTR | Libellé (texte qui décrit) une opération bancaire |
| `DAT_MVT` / `MNT_MVT` | TXN_X_CTR | Date / montant du mouvement (montant **simulé** pour l'exercice) |

In [1]:
import sqlite3                       # module standard Python : base de données SQLite légère
import pandas as pd                  # pour charger les CSV et récupérer les résultats SQL sous forme de tableau
import numpy as np                   # pour simuler les colonnes manquantes de TXN_X_CTR
from pathlib import Path             # gestion de chemins de fichiers indépendante de l'OS

DATA = Path("../data")                                    # dossier contenant les CSV Beobank
# encoding="cp1252" (et non "utf-8") : ces fichiers viennent d'un export Windows/SAS,
# c'est le bon encodage pour lire correctement les lettres accentuées (é, è, ô...)
PARAMS = dict(sep=";", na_values=".", encoding="cp1252")   # "." devient un vrai NULL dès le chargement

# --- Chargement des 5 tables avec Pandas (le plus simple pour ensuite les charger en SQL) ---
ctr     = pd.read_csv(DATA / "CTR.csv",       **PARAMS)   # CTR       = comptes / contrats bancaires
tie     = pd.read_csv(DATA / "TIE.csv",       **PARAMS)   # TIE       = clients ("tiers")
tie_adr = pd.read_csv(DATA / "TIE_ADR.csv",   **PARAMS)   # TIE_ADR   = adresses et coordonnées des clients
txc     = pd.read_csv(DATA / "TIE_X_CTR.csv", **PARAMS)   # TIE_X_CTR = qui possède quel compte
txn     = pd.read_csv(DATA / "TXN_X_CTR.csv", **PARAMS)   # TXN_X_CTR = opérations sur les comptes

# TXN_X_CTR.csv n'a pas de montant de mouvement exploitable -> simulation pédagogique
# (même technique et même graine 42 que dans Cours_SQL.ipynb, pour rester cohérent)
rng = np.random.default_rng(42)                             # générateur aléatoire reproductible (graine 42)
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_CRE_MVT_CPB"])      # date de mouvement
txn["MNT_MVT"] = rng.normal(250, 400, size=len(txn)).round(2)  # montant simulé (moyenne 250€, écart-type 400€)

# --- Connexion SQLite en mémoire (le "moteur SQL" de ce notebook) ---
conn = sqlite3.connect(":memory:")     # base de données temporaire, qui existe le temps de la session

# .to_sql(nom_table, connexion, ...) : écrit un DataFrame Pandas comme une VRAIE table SQL
for nom, df in [("CTR", ctr), ("TIE", tie), ("TIE_ADR", tie_adr),
                ("TIE_X_CTR", txc), ("TXN_X_CTR", txn)]:
    df.to_sql(nom, conn, if_exists="replace", index=False)

print("Tables chargées :", ["CTR", "TIE", "TIE_ADR", "TIE_X_CTR", "TXN_X_CTR"])

Tables chargées : ['CTR', 'TIE', 'TIE_ADR', 'TIE_X_CTR', 'TXN_X_CTR']


---
## Situation 1 — Premier jour : explorer la table des clients

**Mise en situation :** C'est votre premier jour comme data analyst SQL chez Beobank. Votre chef d'équipe vous dit : « Regarde ce qu'il y a dans la table des clients. »

Affichez les 5 premières lignes de la table `TIE` (les clients) avec **`SELECT * ... LIMIT 5`**.

In [2]:
requete = """
-- SELECT * : on demande TOUTES les colonnes
-- FROM TIE : la table des clients ("TIE" = "tiers", le mot bancaire pour "client")
-- LIMIT 5 : on ne veut que les 5 premières lignes, pour un premier aperçu
SELECT * FROM TIE LIMIT 5
"""
print(pd.read_sql(requete, conn))

      IDT_PI        NUM_TIE  COD_TYP_TIE  COD_STA_FED DAT_STA_FED DAT_PRE_CTR  \
0  655010234  2500003178436            2            4  2025-12-22         NaN   
1  655010248  2500003178512            1            3         NaN         NaN   
2  655010249  2500003178544            1            1  2025-11-28  2025-11-28   
3  655010254  2500003178588            1            3         NaN         NaN   
4  655010256  2500003178678            1            3         NaN         NaN   

  DAT_DER_CTR     DAT_NAI COD_LNG_CTR DAT_DCS COD_SEX  
0        None         NaN          FR    None     NaN  
1        None  2004-03-28          FR    None       M  
2        None  1980-01-25          FR    None       M  
3        None  2001-05-18          FR    None       M  
4        None  1975-01-30          FR    None       M  


Comptez le nombre total de clients avec **`COUNT(*)`**.

In [3]:
requete = """
-- COUNT(*) compte le nombre de lignes de la table
-- AS nb_clients : on donne un nom clair à la colonne du résultat (un alias)
SELECT COUNT(*) AS nb_clients FROM TIE
"""
print(pd.read_sql(requete, conn))

   nb_clients
0         100


Listez les langues de contact différentes présentes dans `TIE` avec **`DISTINCT`**.

In [4]:
requete = """
-- COD_LNG_CTR = la langue de contact du client (FR ou NL)
-- DISTINCT enlève les doublons : chaque valeur n'apparaît qu'une fois dans le résultat
SELECT DISTINCT COD_LNG_CTR FROM TIE
"""
print(pd.read_sql(requete, conn))

  COD_LNG_CTR
0          FR
1          NL


**Ce que ça veut dire pour la banque :** On a 100 clients dans la table TIE. Les colonnes viennent du système bancaire d'origine : peu claires au premier regard, mais on va apprendre à les décoder.

---
## Situation 2 — Préparer un mailing marketing

**Mise en situation :** Le service marketing veut savoir : « Est-ce qu'on a surtout des particuliers ou des entreprises ? »

Traduisez le code `COD_TYP_TIE` en texte lisible avec **`CASE WHEN`** (1 = Particulier, 2 = Entreprise).

In [5]:
requete = """
-- COD_TYP_TIE = code type de client (1 = personne physique, 2 = personne morale)
-- CASE WHEN condition THEN valeur ELSE autre_valeur END : le "SI... ALORS... SINON" du SQL
SELECT NUM_TIE,
       CASE WHEN COD_TYP_TIE = 1 THEN 'Particulier' ELSE 'Entreprise' END AS type_client
FROM TIE
LIMIT 5
"""
print(pd.read_sql(requete, conn))

         NUM_TIE  type_client
0  2500003178436   Entreprise
1  2500003178512  Particulier
2  2500003178544  Particulier
3  2500003178588  Particulier
4  2500003178678  Particulier


Comptez le nombre de clients par type avec **`GROUP BY`**.

In [6]:
requete = """
-- GROUP BY type_client : regroupe toutes les lignes qui ont la même valeur de type_client
-- COUNT(*) est alors calculé SÉPARÉMENT pour chaque groupe
SELECT
    CASE WHEN COD_TYP_TIE = 1 THEN 'Particulier' ELSE 'Entreprise' END AS type_client,
    COUNT(*) AS nb_clients
FROM TIE
GROUP BY type_client
"""
print(pd.read_sql(requete, conn))

   type_client  nb_clients
0   Entreprise           1
1  Particulier          99


Faites la même chose sur `COD_LNG_CTR` (la langue de contact).

In [7]:
requete = """
-- même principe : un groupe par langue de contact (FR / NL), et un comptage par groupe
SELECT COD_LNG_CTR, COUNT(*) AS nb_clients
FROM TIE
GROUP BY COD_LNG_CTR
"""
print(pd.read_sql(requete, conn))

  COD_LNG_CTR  nb_clients
0          FR          67
1          NL          33


**Ce que ça veut dire pour la banque :** Le portefeuille est presque entièrement composé de particuliers, majoritairement francophones : le marketing peut écrire l'email en français, avec une version néerlandaise pour les autres.

---
## Situation 3 — Traduire le statut des comptes

**Mise en situation :** Un collègue ouvre la table des comptes (`CTR`) et ne comprend pas les codes : « `COD_ECV_CTR` égal à 4 ou 6, ça veut dire quoi ? »

Affichez `IDT_AC` et `COD_ECV_CTR` traduit en texte avec **`CASE WHEN`** (4 = Actif, 6 = Clôturé).

In [8]:
requete = """
-- IDT_AC = identifiant du compte bancaire
-- COD_ECV_CTR = code d'état du compte : 4 = actif, 6 = clôturé
-- CASE WHEN peut avoir plusieurs WHEN à la suite, comme un SI / SINON SI / SINON
SELECT IDT_AC,
       CASE WHEN COD_ECV_CTR = 4 THEN 'Actif' WHEN COD_ECV_CTR = 6 THEN 'Clôturé' END AS statut
FROM CTR
LIMIT 5
"""
print(pd.read_sql(requete, conn))

        IDT_AC   statut
0  65500004701  Clôturé
1  65500006391  Clôturé
2  65500007774    Actif
3  65500008787    Actif
4  65500014230  Clôturé


Comptez le nombre de comptes actifs vs clôturés, en réutilisant ce `CASE WHEN` dans un **`GROUP BY`**.

In [ ]:
requete = """
-- on regroupe par le statut qu'on vient de calculer, et on compte les comptes de chaque groupe
SELECT
    CASE WHEN COD_ECV_CTR = 4 THEN 'Actif' WHEN COD_ECV_CTR = 6 THEN 'Clôturé' END AS statut,
    COUNT(*) AS nb_comptes
FROM CTR
GROUP BY statut
"""
print(pd.read_sql(requete, conn))

Trouvez les comptes dont le solde `SLD_CTR` est manquant avec **`IS NULL`**.

In [ ]:
requete = """
-- SLD_CTR = solde du compte. IS NULL teste si une valeur est MANQUANTE
-- (on ne peut jamais écrire "= NULL" en SQL, NULL se teste avec IS NULL / IS NOT NULL)
SELECT IDT_AC FROM CTR WHERE SLD_CTR IS NULL LIMIT 5
"""
print(pd.read_sql(requete, conn))

**Ce que ça veut dire pour la banque :** Un peu plus de la moitié des comptes de l'extrait sont encore actifs. Beaucoup de soldes sont NULL : rappel que "." dans le CSV a été transformé en vrai NULL SQL dès le chargement.

---
## Situation 4 — Repérer les comptes en découvert

**Mise en situation :** Le service risque prépare un comité de crédit demain matin. Il a besoin de la liste des comptes dont le solde est négatif.

Sélectionnez les comptes en découvert (`SLD_CTR < 0`) avec **`WHERE`**.

In [ ]:
requete = """
-- WHERE filtre les LIGNES : on ne garde que celles où SLD_CTR est strictement négatif
-- COD_DEV = devise du compte (EUR, USD...)
SELECT IDT_AC, SLD_CTR, COD_DEV FROM CTR WHERE SLD_CTR < 0
"""
print(pd.read_sql(requete, conn))

Triez ces comptes du découvert le plus important au moins important avec **`ORDER BY`**.

In [ ]:
requete = """
-- ORDER BY SLD_CTR ASC : tri croissant (ASC = ascending), donc les valeurs les plus
-- négatives (les pires découverts) apparaissent en premier
SELECT IDT_AC, SLD_CTR FROM CTR WHERE SLD_CTR < 0 ORDER BY SLD_CTR ASC
"""
print(pd.read_sql(requete, conn))

Affichez les 5 plus gros soldes positifs avec `ORDER BY ... DESC` et **`LIMIT`**.

In [ ]:
requete = """
-- DESC (descending) : tri décroissant, les plus gros soldes en premier
-- LIMIT 5 : on garde seulement les 5 premières lignes du résultat trié
SELECT IDT_AC, SLD_CTR FROM CTR ORDER BY SLD_CTR DESC LIMIT 5
"""
print(pd.read_sql(requete, conn))

Calculez le montant total en découvert avec **`SUM()`**.

In [ ]:
requete = """
-- SUM(SLD_CTR) additionne tous les soldes des lignes gardées par le WHERE
-- comme ce sont des soldes négatifs, le résultat est un grand nombre négatif
SELECT SUM(SLD_CTR) AS total_decouvert FROM CTR WHERE SLD_CTR < 0
"""
print(pd.read_sql(requete, conn))

**Ce que ça veut dire pour la banque :** Le service risque a maintenant une liste précise des comptes à surveiller, triée du cas le plus grave au moins grave.

---
## Situation 5 — Protéger les clients âgés

**Mise en situation :** Le service conformité doit surveiller particulièrement les clients de 75 ans et plus.

Calculez l'âge de chaque client en jours avec **`julianday()`** (fonction SQLite qui convertit une date en nombre de jours).

In [ ]:
requete = """
-- DAT_NAI = date de naissance du client
-- julianday(date) transforme une date en nombre de jours depuis une origine fixe
-- la différence entre deux julianday() donne un nombre de JOURS entre les deux dates
-- on divise par 365.25 pour obtenir un âge en ANNÉES
SELECT NUM_TIE, DAT_NAI,
       (julianday('now') - julianday(DAT_NAI)) / 365.25 AS age
FROM TIE
LIMIT 5
"""
print(pd.read_sql(requete, conn))

💡 **Repère Vertica :** En Vertica, on écrirait plutôt `DATEDIFF('year', DAT_NAI, CURRENT_DATE)` ou la fonction dédiée `AGE_IN_YEARS(DAT_NAI)` — `julianday()` est spécifique à SQLite.

Gardez uniquement les clients de 75 ans et plus, en réutilisant ce calcul dans un **`WHERE`**.

In [ ]:
requete = """
-- on répète le même calcul d'âge dans le WHERE, avec la condition >= 75
SELECT NUM_TIE, (julianday('now') - julianday(DAT_NAI)) / 365.25 AS age
FROM TIE
WHERE (julianday('now') - julianday(DAT_NAI)) / 365.25 >= 75
"""
print(pd.read_sql(requete, conn))

Comptez ces clients avec **`COUNT(*)`**.

In [ ]:
requete = """
-- même filtre WHERE que juste au-dessus, mais on ne demande que le COMPTAGE cette fois
SELECT COUNT(*) AS nb_clients_ages
FROM TIE
WHERE (julianday('now') - julianday(DAT_NAI)) / 365.25 >= 75
"""
print(pd.read_sql(requete, conn))

**Ce que ça veut dire pour la banque :** La conformité dispose d'une liste exacte des clients de 75 ans et plus, pour mettre en place le suivi renforcé exigé par la réglementation.

---
## Situation 6 — Vérifier les coordonnées des clients

**Mise en situation :** Avant une campagne email, le marketing veut savoir : « Est-ce qu'on a bien les coordonnées de tout le monde ? »

Comptez les clients qui ONT un email, avec **`IS NOT NULL`**.

In [ ]:
requete = """
-- ADR_EMA = adresse email du client, dans la table TIE_ADR
-- IS NOT NULL teste si la valeur EST renseignée (pas manquante)
SELECT COUNT(*) AS nb_avec_email FROM TIE_ADR WHERE ADR_EMA IS NOT NULL
"""
print(pd.read_sql(requete, conn))

Trouvez les clients sans email NI téléphone (mobile ou fixe), en combinant plusieurs `IS NULL` avec **`AND`**.

In [ ]:
requete = """
-- NUM_TEL_MOB_INL = téléphone mobile, NUM_TEL_DOM_INL = téléphone fixe
-- AND veut dire "et" : les TROIS conditions doivent être vraies en même temps
SELECT NUM_TIE, NOM_TIE
FROM TIE_ADR
WHERE ADR_EMA IS NULL AND NUM_TEL_MOB_INL IS NULL AND NUM_TEL_DOM_INL IS NULL
"""
print(pd.read_sql(requete, conn))

Recherchez les clients dont l'email contient `"gmail"` avec **`LIKE`**.

In [ ]:
requete = """
-- LIKE recherche un motif dans du texte. Le symbole % veut dire
-- "n'importe quel texte avant/après" -> ici, tout email qui contient "gmail" quelque part
SELECT NUM_TIE, ADR_EMA FROM TIE_ADR WHERE ADR_EMA LIKE '%gmail%'
"""
print(pd.read_sql(requete, conn))

💡 **Repère Vertica :** `LIKE` est **sensible à la casse** sur Vertica (contrairement à SQLite). Pour ignorer la casse sur Vertica, utiliser `ILIKE` à la place de `LIKE`.

**Ce que ça veut dire pour la banque :** La majorité des clients ont un email connu, mais certains sont totalement injoignables : pour eux, la campagne email ne fonctionnera pas.

---
## Situation 7 — Enquêter sur un client qui a beaucoup de comptes

**Mise en situation :** Le directeur régional remarque qu'un client semble avoir énormément de comptes. Il vous demande de vérifier.

Comptez le nombre de comptes par client avec **`GROUP BY`** sur `TIE_X_CTR`.

In [ ]:
requete = """
-- TIE_X_CTR = la table qui relie chaque client (NUM_TIE) à ses comptes (IDT_AC)
-- un groupe par client, et on compte les lignes (donc les comptes) de chaque groupe
SELECT NUM_TIE, COUNT(*) AS nb_comptes
FROM TIE_X_CTR
GROUP BY NUM_TIE
ORDER BY nb_comptes DESC
"""
print(pd.read_sql(requete, conn))

Gardez uniquement les clients qui ont plus de 5 comptes, avec **`HAVING`** (le `WHERE` des groupes).

In [ ]:
requete = """
-- WHERE filtre les LIGNES avant regroupement ; HAVING filtre les GROUPES après le COUNT
-- on ne peut pas écrire WHERE COUNT(*) > 5 : il FAUT HAVING pour filtrer un résultat agrégé
SELECT NUM_TIE, COUNT(*) AS nb_comptes
FROM TIE_X_CTR
GROUP BY NUM_TIE
HAVING COUNT(*) > 5
ORDER BY nb_comptes DESC
"""
print(pd.read_sql(requete, conn))

Trouvez les clients de `TIE` absents de `TIE_X_CTR` (aucun compte) avec **`NOT IN`** et une sous-requête.

In [ ]:
requete = """
-- (SELECT NUM_TIE FROM TIE_X_CTR) est une SOUS-REQUÊTE : elle renvoie la liste de
-- tous les NUM_TIE qui ONT un compte
-- NOT IN (...) garde les clients dont le NUM_TIE n'est PAS dans cette liste
SELECT NUM_TIE FROM TIE
WHERE NUM_TIE NOT IN (SELECT NUM_TIE FROM TIE_X_CTR)
"""
print(pd.read_sql(requete, conn))

**Ce que ça veut dire pour la banque :** Un seul client concentre une grande partie des comptes de l'extrait. On voit aussi que beaucoup de clients de TIE n'ont aucun compte dans TIE_X_CTR : deux exports qui ne couvrent pas le même périmètre.

---
## Situation 8 — Répondre à un client qui conteste des frais

**Mise en situation :** Un client appelle, énervé : « Il y a plein de prélèvements que je ne comprends pas sur mon compte `65500477817` ! »

Affichez toutes les opérations de ce compte, triées par date, avec **`WHERE`** et `ORDER BY`.

In [ ]:
requete = """
-- LIB_OPE_INL_1 = libellé qui décrit l'opération, MNT_MVT = montant simulé du mouvement
-- WHERE IDT_AC = ... : uniquement les opérations de ce compte précis
-- ORDER BY DAT_MVT : de la plus ancienne à la plus récente
SELECT DAT_MVT, LIB_OPE_INL_1, MNT_MVT
FROM TXN_X_CTR
WHERE IDT_AC = 65500477817
ORDER BY DAT_MVT
"""
print(pd.read_sql(requete, conn))

Filtrez seulement les frais avec **`LIKE`** sur le libellé.

In [ ]:
requete = """
-- LIKE '%frais%' : le libellé contient le mot "frais" n'importe où dans le texte
SELECT DAT_MVT, LIB_OPE_INL_1, MNT_MVT
FROM TXN_X_CTR
WHERE IDT_AC = 65500477817 AND LIB_OPE_INL_1 LIKE '%frais%'
ORDER BY DAT_MVT
"""
print(pd.read_sql(requete, conn))

Calculez le total des frais de ce compte avec **`SUM()`**.

In [ ]:
requete = """
-- SUM(MNT_MVT) additionne les montants simulés de toutes les opérations de frais trouvées
SELECT SUM(MNT_MVT) AS total_frais
FROM TXN_X_CTR
WHERE IDT_AC = 65500477817 AND LIB_OPE_INL_1 LIKE '%frais%'
"""
print(pd.read_sql(requete, conn))

**Ce que ça veut dire pour la banque :** Le conseiller peut désormais distinguer clairement les frais des autres opérations, avec leurs dates et leur montant total, pour répondre précisément au client.

---
## Situation 9 — Prévoir l'activité du centre d'appel

**Mise en situation :** Le service RH doit décider combien de conseillers embaucher. Il demande quel mois a le plus d'opérations.

Regroupez les opérations par mois avec **`strftime('%Y-%m', ...)`** (fonction SQLite qui formate une date).

In [ ]:
requete = """
-- strftime('%Y-%m', DAT_MVT) transforme chaque date en "année-mois" (ex: 2026-04)
-- GROUP BY mois : un groupe par mois, COUNT(*) compte les opérations de chaque mois
SELECT strftime('%Y-%m', DAT_MVT) AS mois, COUNT(*) AS nb_operations
FROM TXN_X_CTR
GROUP BY mois
ORDER BY mois
"""
print(pd.read_sql(requete, conn))

💡 **Repère Vertica :** En Vertica, on utiliserait plutôt `TO_CHAR(DAT_MVT, 'YYYY-MM')` ou `DATE_TRUNC('month', DAT_MVT)` — `strftime()` est spécifique à SQLite.

Affichez le mois le plus chargé en premier avec **`ORDER BY ... DESC`**.

In [ ]:
requete = """
-- ORDER BY nb_operations DESC : tri décroissant sur le comptage
-- LIMIT 1 : on ne garde que le tout premier -> le mois le plus chargé
SELECT strftime('%Y-%m', DAT_MVT) AS mois, COUNT(*) AS nb_operations
FROM TXN_X_CTR
GROUP BY mois
ORDER BY nb_operations DESC
LIMIT 1
"""
print(pd.read_sql(requete, conn))

**Ce que ça veut dire pour la banque :** Le RH sait sur quel mois concentrer le renfort d'équipe.

---
## Situation 10 — Construire une fiche client à 360°

**Mise en situation :** Un conseiller a un rendez-vous avec un client important. Il veut voir en une fois tous ses comptes et leur solde.

Fusionnez `TIE`, `TIE_X_CTR` et `CTR` avec **`JOIN`**, sur les bonnes colonnes communes.

In [ ]:
requete = """
-- JOIN table ON condition : rapproche deux tables qui ont une colonne en commun
-- t, x, c sont des ALIAS (raccourcis) pour TIE, TIE_X_CTR et CTR
-- ici on enchaîne deux JOIN : TIE -> TIE_X_CTR (par NUM_TIE) -> CTR (par IDT_AC)
SELECT t.NUM_TIE, c.IDT_AC, c.SLD_CTR, c.COD_DEV
FROM TIE t
JOIN TIE_X_CTR x ON t.NUM_TIE = x.NUM_TIE
JOIN CTR c ON x.IDT_AC = c.IDT_AC
LIMIT 5
"""
print(pd.read_sql(requete, conn))

Calculez le solde total par client avec **`GROUP BY`** et `SUM()`, pour le client `2900000004654`.

In [ ]:
requete = """
-- on ajoute une clause GROUP BY pour agréger tous les comptes d'un même client
-- WHERE filtre AVANT le regroupement : seulement les lignes de ce client précis
SELECT t.NUM_TIE, SUM(c.SLD_CTR) AS solde_total, COUNT(c.IDT_AC) AS nb_comptes
FROM TIE t
JOIN TIE_X_CTR x ON t.NUM_TIE = x.NUM_TIE
JOIN CTR c ON x.IDT_AC = c.IDT_AC
WHERE t.NUM_TIE = 2900000004654
GROUP BY t.NUM_TIE
"""
print(pd.read_sql(requete, conn))

Créez une **`VIEW`** (vue) réutilisable pour cette fiche 360°, sans le filtre `WHERE`.

In [ ]:
conn.executescript("""
-- CREATE VIEW nom AS requête : enregistre la requête sous un nom, comme une table virtuelle
-- ensuite, "SELECT * FROM vue_360" relance la requête complète sans la réécrire
-- DROP VIEW IF EXISTS : supprime la vue si elle existe déjà (pour pouvoir relancer la cellule)
DROP VIEW IF EXISTS vue_360;
CREATE VIEW vue_360 AS
SELECT t.NUM_TIE, SUM(c.SLD_CTR) AS solde_total, COUNT(c.IDT_AC) AS nb_comptes
FROM TIE t
JOIN TIE_X_CTR x ON t.NUM_TIE = x.NUM_TIE
JOIN CTR c ON x.IDT_AC = c.IDT_AC
GROUP BY t.NUM_TIE;
""")
conn.commit()
print("Vue créée.")

Interrogez la vue pour retrouver les 5 plus gros patrimoines avec un simple **`SELECT`**.

In [ ]:
requete = """
-- la vue vue_360 se lit comme une table normale, alors qu'elle rejoue en fait
-- la requête à 3 tables définie juste au-dessus
SELECT * FROM vue_360 ORDER BY solde_total DESC LIMIT 5
"""
print(pd.read_sql(requete, conn))

**Ce que ça veut dire pour la banque :** Le conseiller obtient, en une seule requête réutilisable (la vue), la situation complète de n'importe quel client : nombre de comptes et solde total.